### causal attention

In [1]:
import torch

inputs = torch.tensor([[0.72, 0.45, 0.31], # Dream 
                    [0.75, 0.20, 0.55], # big 
                    [0.30, 0.80, 0.40], # and 
                    [0.85, 0.35, 0.60], # work 
                    [0.55, 0.15, 0.75], # for 
                    [0.25, 0.20, 0.85]]) # it 
words = ['Dream','big','and','work','for','it']

In [11]:
import torch.nn as nn
from math import sqrt
class Self_attention(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out))
    
    def forward(self,x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        att_scores = queries @ keys.T
        att_weights = torch.softmax(att_scores/sqrt(keys.shape[-1]),dim = -1)

        context_vec = att_weights @ values 
        return att_scores,att_weights,context_vec



In [12]:
torch.manual_seed(123)
sa_v1 = Self_attention(d_in = inputs.shape[-1], d_out = 2)
attention_scores,attention_weights,output = sa_v1(inputs)
print(output)

tensor([[0.2273, 0.7361],
        [0.2274, 0.7362],
        [0.2276, 0.7363],
        [0.2280, 0.7368],
        [0.2275, 0.7362],
        [0.2275, 0.7360]], grad_fn=<MmBackward0>)


In [5]:
context_length = inputs.shape[0]
mask_simple = torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [9]:
attention_mask_before_normalization = attention_weights * mask_simple
attention_mask_before_normalization

tensor([[0.1536, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1531, 0.1528, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1525, 0.1521, 0.1884, 0.0000, 0.0000, 0.0000],
        [0.1505, 0.1501, 0.1915, 0.1737, 0.0000, 0.0000],
        [0.1530, 0.1524, 0.1881, 0.1724, 0.1625, 0.0000],
        [0.1538, 0.1530, 0.1875, 0.1719, 0.1625, 0.1713]],
       grad_fn=<MulBackward0>)

In [10]:
row_sums = attention_mask_before_normalization.sum(dim = 1, keepdim = True)
attention_mask_after_normalization = attention_mask_before_normalization/row_sums
print(attention_mask_after_normalization)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5005, 0.4995, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3093, 0.3086, 0.3821, 0.0000, 0.0000, 0.0000],
        [0.2260, 0.2255, 0.2877, 0.2609, 0.0000, 0.0000],
        [0.1847, 0.1840, 0.2271, 0.2081, 0.1961, 0.0000],
        [0.1538, 0.1530, 0.1875, 0.1719, 0.1625, 0.1713]],
       grad_fn=<DivBackward0>)


### making upper triangle of attention_scores to -infinity

In [14]:
print(attention_scores)

tensor([[0.6807, 0.6795, 0.9526, 0.8454, 0.7654, 0.8359],
        [0.7021, 0.6990, 0.9867, 0.8707, 0.7880, 0.8624],
        [0.7350, 0.7315, 1.0337, 0.9113, 0.8248, 0.9029],
        [0.8436, 0.8402, 1.1848, 1.0464, 0.9471, 1.0361],
        [0.7080, 0.7025, 1.0003, 0.8764, 0.7929, 0.8699],
        [0.6680, 0.6606, 0.9486, 0.8254, 0.7465, 0.8210]],
       grad_fn=<MmBackward0>)


In [16]:
mask = torch.triu(torch.ones(context_length,context_length),diagonal = 1)
print(mask)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


In [22]:
# ones are converted to -infinite
print(mask)
print(attention_scores)
attention_scores = attention_scores/(inputs.shape[-1] ** 0.5 )
masked = attention_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)



tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.6807, 0.6795, 0.9526, 0.8454, 0.7654, 0.8359],
        [0.7021, 0.6990, 0.9867, 0.8707, 0.7880, 0.8624],
        [0.7350, 0.7315, 1.0337, 0.9113, 0.8248, 0.9029],
        [0.8436, 0.8402, 1.1848, 1.0464, 0.9471, 1.0361],
        [0.7080, 0.7025, 1.0003, 0.8764, 0.7929, 0.8699],
        [0.6680, 0.6606, 0.9486, 0.8254, 0.7465, 0.8210]],
       grad_fn=<MmBackward0>)
tensor([[0.3930,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4053, 0.4036,   -inf,   -inf,   -inf,   -inf],
        [0.4244, 0.4223, 0.5968,   -inf,   -inf,   -inf],
        [0.4870, 0.4851, 0.6841, 0.6041,   -inf,   -inf],
        [0.4088, 0.4056, 0.5775, 0.5060, 0.4578,   -inf],
        [0.3857, 0.3814, 0.5477, 0.4765, 0.4310, 0.4740]],
       grad_fn=<MaskedFillBackward0>)


In [23]:
# apply softmax row-wise
attention_weights = torch.softmax(masked,dim= -1)
print(attention_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5004, 0.4996, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3139, 0.3132, 0.3729, 0.0000, 0.0000, 0.0000],
        [0.2304, 0.2300, 0.2806, 0.2590, 0.0000, 0.0000],
        [0.1875, 0.1869, 0.2220, 0.2067, 0.1969, 0.0000],
        [0.1561, 0.1555, 0.1836, 0.1710, 0.1634, 0.1705]],
       grad_fn=<SoftmaxBackward0>)
